**Implementing Double Q-Learning in frozen lake environment**

In [37]:
import numpy as np
import gymnasium as gym
import time

In [38]:
# Create the FrozenLake environment (Not slippery!!)
env = gym.make(
    "FrozenLake-v1", is_slippery=False, render_mode=None
)  # Uses a separate window for rendering

In [56]:
# Initialize Q-table with zeros
state_size = env.observation_space.n  # Number of states (16 for 4x4 grid)
action_size = env.action_space.n  # Number of actions (4: left, down, right, up)
Q_table1 = np.zeros((state_size, action_size))
Q_table2 = np.zeros((state_size, action_size))

print(f"State size: {state_size}, Action size: {action_size}")

# Q-learning parameters
learning_rate = 0.1  # Alpha: Learning rate
discount_factor = 0.95  # Gamma: Discount factor for future rewards
epsilon = 1.0  # Epsilon: Initial exploration rate
epsilon_decay = 0.995  # Epsilon decay factor
min_epsilon = 0.01  # Minimum exploration rate
num_episodes = 2000  # Total episodes for training
max_steps = 100  # Max steps per episode

State size: 16, Action size: 4


In [57]:
for episode in range(num_episodes):
    state, _ = env.reset()  # Initialize the environment and get the starting state.
    # state = 0
    done = False

    for step in range(max_steps):
        # Select an action using the epsilon-greedy policy
        if np.random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q_table1[state, :] + Q_table2[state, :])

        # Take the action and observe the reward and the next state.
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        if terminated and next_state != 15:
            reward = -1

        if truncated:
            reward = -0.1

        # Update the Q-table.
        if np.random.uniform(0, 1) < 0.5:
            Q_table1[state, action] = Q_table1[state, action] + learning_rate * (
                reward
                + discount_factor
                * Q_table2[next_state, np.argmax(Q_table1[next_state, :])]
                - Q_table1[state, action]
            )
        else:
            Q_table2[state, action] = Q_table2[state, action] + learning_rate * (
                reward
                + discount_factor
                * Q_table1[next_state, np.argmax(Q_table2[next_state, :])]
                - Q_table2[state, action]
            )

        state = next_state  # Move to next state

        if done:
            break  # End episode if goal reached or fallen in hole

    # Decay epsilon (reduce exploration over time)
    epsilon = max(min_epsilon, epsilon * epsilon_decay)

print("Training complete!")

Training complete!


In [58]:
# Test the trained agent
test_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")
num_test_episodes = 5

In [59]:
for episode in range(num_test_episodes):
    state, _ = test_env.reset()
    done = False
    print(f"\nEpisode {episode+1}:")
    time.sleep(1)

    for step in range(100):
        action = np.argmax(Q_table1[state, :] + Q_table2[state, :])
        next_state, reward, terminated, truncated, _ = test_env.step(action)
        done = terminated or truncated
        time.sleep(0.5)

        if done:
            if reward == 1:
                print("🎉 Goal reached!")
            else:
                print("💀 Fell into a hole.")
            break

        state = next_state  # Move to next state

test_env.close()


Episode 1:
🎉 Goal reached!

Episode 2:


KeyboardInterrupt: 